# Echocardiography Risk Prediction Model - Training Script

This notebook generates a realistic synthetic dataset of Echocardiography metrics (LVEF, LVEDD, Wall Motion Abnormalities, Mitral Regurgitation) and trains an XGBoost model to predict the probability of three specific cardiovascular diseases:
1. **Heart Failure**
2. **Coronary Artery Disease (CAD)**
3. **Cardiomyopathy**

## Instructions:
1. Run all cells in this notebook.
2. At the end, it will automatically download a file named `echo_xgboost.pkl` to your computer.
3. Move this downloaded `.pkl` file into your project's `models/` folder and paste the path to the AI Agent.

In [ ]:
!pip install xgboost scikit-learn pandas numpy

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import pickle
from google.colab import files

### 1. Generate Synthetic Echo Dataset
We create rules-based synthetic data reflecting clinical realities:
- Low LVEF (<40%) strongly correlates with Heart Failure and Cardiomyopathy.
- High LVEDD (>60 mm) indicates dilated cardiomyopathy or advanced heart failure.
- Wall Motion Abnormalities strongly indicate CAD.

In [ ]:
np.random.seed(42)
n_samples = 5000

# Features
lvef = np.random.normal(loc=55, scale=12, size=n_samples).clip(15, 75)
lvedd = np.random.normal(loc=50, scale=8, size=n_samples).clip(35, 80)
# Wall Motion: 0 = Normal, 1 = Hypokinesia, 2 = Akinesia
wall_motion = np.random.choice([0, 1, 2], p=[0.7, 0.2, 0.1], size=n_samples)
# Mitral Regurgitation: 0 = None, 1 = Mild, 2 = Moderate, 3 = Severe
mitral_regurg = np.random.choice([0, 1, 2, 3], p=[0.5, 0.3, 0.15, 0.05], size=n_samples)

X = pd.DataFrame({
    'LVEF': lvef,
    'LVEDD': lvedd,
    'WallMotion': wall_motion,
    'MitralRegurgitation': mitral_regurg
})

# Targets (Probabilities based on heuristic rules, then converted to binary classes)
prob_hf = 1 / (1 + np.exp(-(-5 + 0.1*(60 - lvef) + 0.1*(lvedd - 50) + 0.5*mitral_regurg)))
prob_cad = 1 / (1 + np.exp(-(-4 + 1.5*wall_motion + 0.02*(60 - lvef))))
prob_cmp = 1 / (1 + np.exp(-(-6 + 0.15*(55 - lvef) + 0.2*(lvedd - 55))))

y_hf = (np.random.rand(n_samples) < prob_hf).astype(int)
y_cad = (np.random.rand(n_samples) < prob_cad).astype(int)
y_cmp = (np.random.rand(n_samples) < prob_cmp).astype(int)

Y = pd.DataFrame({
    'HeartFailure': y_hf,
    'CAD': y_cad,
    'Cardiomyopathy': y_cmp
})

print("Dataset Shape:", X.shape)
print("Target distributions:\n", Y.mean())

### 2. Train Multi-Output XGBoost Model

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# MultiOutputRegressor wrapper using MultiOutputClassifier isn't necessary for probabilities if we train 3 separate models or one multi-target 
# We will train a dictionary of 3 models for simplicity and robust probability extraction

models = {}
for col in Y.columns:
    print(f"Training model for {col}...")
    model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, Y_train[col])
    
    # Evaluate
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"  Accuracy: {accuracy_score(Y_test[col], preds):.3f} | AUC: {roc_auc_score(Y_test[col], probs):.3f}")
    
    models[col] = model

print("\nAll models trained successfully!")

### 3. Save & Download Model

In [ ]:
# Pack the models together with the feature list
export_package = {
    'features': ['LVEF', 'LVEDD', 'WallMotion', 'MitralRegurgitation'],
    'models': models
}

filename = 'echo_xgboost.pkl'
with open(filename, 'wb') as f:
    pickle.dump(export_package, f)

print(f"Saved to {filename}. Triggering download...")
files.download(filename)